# 02 · Colab A100/H100 无头批处理量产（断点续传 + 跑完自动释放防烧卡）

- 读取 `/MyDrive/AI_Video/tasks/tasks.csv`，逐条调用本地无头 ComfyUI API 生成视频。
- 每条视频生成完毕立即落盘 Google Drive 并写入 `manifest.jsonl`，断线重跑自动跳过已完成任务。
- 默认开启 `AUTO_DISCONNECT = True`，整批任务完成后自动释放 H100，最大化节省算力单元 (CU)。

In [ ]:
#@title 1. 挂载 Google Drive、拉取代码并检查 GPU
REPO_URL = "https://github.com/WSQ233/AI-Video.git" #@param {type:"string"}
REPO_DIR = "/content/AI-Video" #@param {type:"string"}

!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess
from pathlib import Path
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# 如果 Drive 中还没有 tasks.csv，自动复制一份示例模板
sample_dst = Path('/content/drive/MyDrive/AI_Video/tasks/tasks_sample.csv')
sample_dst.parent.mkdir(parents=True, exist_ok=True)
if not sample_dst.exists():
    shutil.copy2(f'{REPO_DIR}/tasks/tasks_sample.csv', sample_dst)
    print(f'已自动生成示例任务表: {sample_dst}')


In [ ]:
#@title 2. 一键装机并预热模型包到本地 SSD
ACTIVE_PACKS = "minimax_h3_fl2va_bf16 minimax_h3_turbo_lora" #@param {type:"string"}
MODEL_LOAD_MODE = "copy_to_local" #@param ["copy_to_local", "symlink"]
INSTALL_SAGEATTENTION = False #@param {type:"boolean"}
ENABLE_LTX_NODE = False #@param {type:"boolean"}

import subprocess
cmd = [
    'python', f'{REPO_DIR}/scripts/setup_env.py',
    '--repo-root', REPO_DIR,
    '--mode', MODEL_LOAD_MODE,
    '--packs', *ACTIVE_PACKS.split(),
]
if INSTALL_SAGEATTENTION:
    cmd.append('--sageattention')
if ENABLE_LTX_NODE:
    cmd.append('--enable-ltx')

subprocess.run(cmd, check=True)


In [ ]:
#@title 3. 启动无头 ComfyUI 并执行批处理队列
TASKS_CSV = "/content/drive/MyDrive/AI_Video/tasks/tasks.csv" #@param {type:"string"}
WORKFLOWS_DIR = "/content/drive/MyDrive/AI_Video/workflows" #@param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/AI_Video/outputs" #@param {type:"string"}
BATCH_NAME = "" #@param {type:"string"}
AUTO_DISCONNECT = True #@param {type:"boolean"}

import json, shlex, subprocess, time
from pathlib import Path

launch_info = json.loads(Path('/content/ComfyUI/launch_info.json').read_text(encoding='utf-8'))
comfy_dir = launch_info['comfyui_dir']
comfy_args = shlex.split(launch_info['comfyui_args'])
log_file = Path('/content/drive/MyDrive/AI_Video/logs/comfyui_headless.log')
log_file.parent.mkdir(parents=True, exist_ok=True)

log_fp = log_file.open('a', encoding='utf-8')
server_proc = subprocess.Popen(
    ['python', 'main.py', *comfy_args],
    cwd=comfy_dir,
    stdout=log_fp,
    stderr=subprocess.STDOUT,
)

try:
    runner_cmd = [
        'python', f'{REPO_DIR}/scripts/batch_runner.py',
        '--tasks-csv', TASKS_CSV,
        '--workflows-dir', WORKFLOWS_DIR,
        '--output-dir', OUTPUT_DIR,
    ]
    if BATCH_NAME.strip():
        runner_cmd.extend(['--batch-name', BATCH_NAME.strip()])
    subprocess.run(runner_cmd, check=False)
finally:
    server_proc.terminate()
    log_fp.close()
    # 给 Google Drive FUSE 留足落盘同步缓冲
    !sync
    time.sleep(15)
    if AUTO_DISCONNECT:
        print('[自动止损] 批处理完成并已同步，即将自动释放 Colab H100 运行时...')
        from google.colab import runtime
        runtime.unassign()
